In [5]:
import sys
from pathlib import Path
import torch
from torch import nn

# Project paths
PROJECT_DIR = Path(r"D:\multimodal_emotion_project")
SRC_DIR = PROJECT_DIR / "src"

# Allow Python to find models.py
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import existing architectures
from models import AudioCNN, VisualCNN, MultimodalModel

DEVICE = torch.device("cpu")
NUM_CLASSES = 8

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)
print("Model classes imported successfully!")

PyTorch version: 2.14.1+cpu
Device: cpu
Model classes imported successfully!


In [6]:
# Create the audio model
audio_model = AudioCNN(num_classes=NUM_CLASSES).to(DEVICE)
audio_model.eval()

# Simulate a batch of 2 audio samples
dummy_audio = torch.randn(2, 1, 96, 251)

# Forward pass without training
with torch.no_grad():
    audio_output = audio_model(dummy_audio)

print("Audio input shape:", tuple(dummy_audio.shape))
print("Audio output shape:", tuple(audio_output.shape))

assert audio_output.shape == (2, NUM_CLASSES)
print("Audio CNN test passed!")

Audio input shape: (2, 1, 96, 251)
Audio output shape: (2, 8)
Audio CNN test passed!


In [7]:
# Disable pretrained weights to avoid downloading them for this test
visual_model = VisualCNN(
    num_classes=NUM_CLASSES,
    pretrained=False
).to(DEVICE)

visual_model.eval()

# Simulate a batch of 2 videos
dummy_video = torch.randn(2, 8, 3, 224, 224)

with torch.no_grad():
    visual_output = visual_model(dummy_video)

print("Video input shape:", tuple(dummy_video.shape))
print("Visual output shape:", tuple(visual_output.shape))

assert visual_output.shape == (2, NUM_CLASSES)
print("Visual CNN test passed!")

Video input shape: (2, 8, 3, 224, 224)
Visual output shape: (2, 8)
Visual CNN test passed!


In [8]:
fusion_modes = [
    "early",
    "late",
    "gated",
    "early_dropout",
    "gated_dropout",
]

# Simulated inputs
dummy_audio = torch.randn(2, 1, 96, 251)
dummy_video = torch.randn(2, 8, 3, 224, 224)

for mode in fusion_modes:
    print(f"\nTesting {mode} fusion...")

    model = MultimodalModel(
        mode=mode,
        num_classes=NUM_CLASSES,
        pretrained=False,
        modality_dropout=mode.endswith("_dropout"),
    ).to(DEVICE)

    model.eval()

    with torch.no_grad():
        output = model(dummy_audio, dummy_video)

    print("Output shape:", tuple(output.shape))

    assert output.shape == (2, NUM_CLASSES), (
        f"{mode} produced an unexpected output shape!"
    )

    del model

print("\nAll five fusion configurations passed!")


Testing early fusion...
Output shape: (2, 8)

Testing late fusion...
Output shape: (2, 8)

Testing gated fusion...
Output shape: (2, 8)

Testing early_dropout fusion...
Output shape: (2, 8)

Testing gated_dropout fusion...
Output shape: (2, 8)

All five fusion configurations passed!
